# MultiMerge Genomics on Colab

Tools for the most common everyday problems with genomic files, running on the Colab machine — you do not need to install anything on your computer.

| What you want to do | Section |
| :--- | :--- |
| Find out **what is wrong** with a file (VCF, BED, GFF/GTF, SAM, FASTA, FASTQ) | 3 |
| Replace `1` with `chr1` (or the other way round) in the whole file | 4 |
| Check whether two files use the **same reference genome** (GRCh37 × GRCh38) | 5 |
| **Sort** VCF, BED, GFF or SAM files of any size | 6 |
| **Merge** and **intersect** BED intervals | 7 |

Run the cells from top to bottom (▶ on the left of each one). Each cell shows a form; the code is hidden, and a double click on the cell shows exactly what it does — it is plain Python, and you can copy it into your own script.

Files of any size are processed as a stream or in sorted runs on disk: the limit is the machine's disk, not its memory.

## 1. Install

In [ ]:
#@title Install MultiMerge Genomics { display-mode: "form" }
#@markdown Where to install from: the address (URL) or the path of a `.whl` file. If you leave it blank, the cell uses whatever is already installed, then looks for a `.whl` in `/content` and, if it finds none, asks you to upload the file.
SOURCE = ""  #@param {type:"string"}

import glob
import importlib.util
import subprocess
import sys

# TEACHER: once the package is published, just change the value above -- to
# the name on PyPI ("multimerge-genomics") or to the URL of the .whl in the
# GitHub Release -- and students will not need to upload any file. Until you
# have registered the name on PyPI yourself, do NOT put it here: someone else
# could register it and the notebook would install their package.
target = SOURCE.strip()
if not target and importlib.util.find_spec("multimerge_genomics") is None:
    found = sorted(glob.glob("/content/multimerge_genomics-*.whl") + glob.glob("/content/wheels/multimerge_genomics-*.whl"))
    if found:
        target = found[-1]
    else:
        from google.colab import files  # only exists inside Colab
        print("Upload the file multimerge_genomics-...whl")
        target = next(iter(files.upload()))
if target:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps", target], check=True)

import multimerge_genomics as mmg

info = mmg.engine_info()
print(f"MultiMerge Genomics {info['version']} installed.")
print(f"CPU: {info['threads']} cores.")
if info["gpu"]:
    free, total = info["gpu_memory_mb"]
    print(f"GPU: available ({total / 1024:.0f} GB). The k-mer modules will use it.")
else:
    print("GPU: none (everything runs on the CPU, with the same result). To turn it on: Runtime → Change runtime type → GPU.")

## 2. Example files

This cell creates an `examples/` folder with small files — some with defects planted on purpose — for you to try out the sections that follow. To use **your own** files, upload them with the folder icon on the left (or mount Google Drive) and change the path in the forms.

In [ ]:
#@title Create the example files { display-mode: "form" }
import os
import random

os.makedirs("examples", exist_ok=True)
random.seed(1)

# Real chromosome sizes in the two human assemblies.
GRCH38 = {"1": 248956422, "2": 242193529, "X": 156040895, "MT": 16569}
GRCH37 = {"1": 249250621, "2": 243199373, "X": 155270560, "MT": 16569}


def chrom_name(name, prefix):
    # In the "chr" convention the mitochondrion is chrM, not chrMT.
    return prefix + ("M" if prefix and name == "MT" else name)


def vcf_header(sizes, prefix=""):
    lines = ["##fileformat=VCFv4.2"]
    lines += [f"##contig=<ID={chrom_name(name, prefix)},length={size}>" for name, size in sizes.items()]
    lines += ['##INFO=<ID=DP,Number=1,Type=Integer,Description="Depth">',
              "#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO"]
    return lines


def variants(sizes, n, prefix=""):
    out = []
    for name, size in sizes.items():
        for pos in sorted(random.sample(range(1, min(size, 5_000_000)), n)):
            ref, alt = random.sample("ACGT", 2)
            out.append(f"{chrom_name(name, prefix)}\t{pos}\trs{random.randrange(10**6)}\t{ref}\t{alt}\t{random.randrange(10, 99)}\tPASS\tDP={random.randrange(5, 80)}")
    return out


def write(name, lines):
    with open(f"examples/{name}", "w") as f:
        f.write("\n".join(lines) + "\n")


# 1) A correct VCF, GRCh38, names without "chr".
good = variants(GRCH38, 2000)
write("patient_grch38.vcf", vcf_header(GRCH38) + good)

# 2) The same kind of file in GRCh37, with "chr1" names.
write("cohort_grch37_chr.vcf", vcf_header(GRCH37, "chr") + variants(GRCH37, 2000, "chr"))

# 3) A shuffled VCF with planted defects.
bad = good[:]
random.shuffle(bad)
bad[10] = bad[10].replace("\t", " ", 1)                   # a space in place of a TAB
fields = bad[20].split("\t"); fields[1] = "12mil"; bad[20] = "\t".join(fields)   # a position that is not a number
fields = bad[30].split("\t"); fields[3] = "N?"; bad[30] = "\t".join(fields)      # an invalid base
fields = bad[40].split("\t"); fields[0] = "chr" + fields[0]; bad[40] = "\t".join(fields)  # a name in the other convention
write("with_defects.vcf", vcf_header(GRCH38) + bad)

# 4) Two BED files: genes (with "chr") and regions of interest.
genes = [f"chr{random.choice('12X')}\t{a}\t{a + random.randrange(500, 20000)}\tgene{i}" for i, a in enumerate(random.sample(range(1, 4_000_000), 800))]
targets = [f"chr{random.choice('12')}\t{a}\t{a + random.randrange(200, 3000)}\ttarget{i}" for i, a in enumerate(random.sample(range(1, 4_000_000), 600))]
write("genes.bed", genes)
write("targets.bed", targets)

for name in sorted(os.listdir("examples")):
    print(f"examples/{name}  ({os.path.getsize('examples/' + name) / 1024:.0f} KB)")

## 3. What is wrong with this file?

The validator reads the whole file (of any size) and answers in plain language: for each kind of problem, the **line** of the first occurrence, what is wrong, how many times it repeats and how to fix it.

In [ ]:
#@title Validate a file { display-mode: "form" }
FILE = "examples/with_defects.vcf"  #@param {type:"string"}
#@markdown Format: leave `auto` to work it out from the file extension.
FORMAT = "auto"  #@param ["auto", "vcf", "bed", "gff", "sam", "fasta", "fastq"]
#@markdown For a quick check of a huge file, validate only the beginning (0 = whole file).
ONLY_FIRST_LINES = 0  #@param {type:"integer"}

import multimerge_genomics as mmg

report = mmg.validate_file(
    FILE,
    format=None if FORMAT == "auto" else FORMAT,
    max_lines=ONLY_FIRST_LINES or None,
)
print(report["text"])

## 4. `chr1` or `1`? Making chromosome names consistent

Half of the tools write `chr1`, `chrX`, `chrM` (the UCSC convention); the other half write `1`, `X`, `MT` (Ensembl/NCBI). Files that use different conventions do not match each other — and many tools give no warning: they just return an empty result. This cell rewrites the whole file in a single convention, header included.

In [ ]:
#@title Normalize chromosome names { display-mode: "form" }
INPUT = "examples/patient_grch38.vcf"  #@param {type:"string"}
OUTPUT = "examples/patient_grch38_chr.vcf"  #@param {type:"string"}
STYLE = "chr"  #@param ["chr", "no_chr"]

import multimerge_genomics as mmg

r = mmg.normalize_chromosomes(INPUT, OUTPUT, style=STYLE)
print(f"{r['records']} records read, {r['changed']} changed. Written to {OUTPUT}")
for before, after in r["converted"]:
    print(f"   {before:>6} -> {after}")
if r["kept"]:
    print("Names left as they were (not known chromosomes):", ", ".join(r["kept"][:20]))

## 5. Are these two files from the same reference genome?

GRCh37 (hg19) coordinates are **not valid** in GRCh38 (hg38). Combining files from different assemblies gives a wrong result with no error message at all. This cell compares what each file declares (the sizes of its sequences) and gives the verdict.

In [ ]:
#@title Compare the reference genome of two files { display-mode: "form" }
FILE_A = "examples/patient_grch38.vcf"  #@param {type:"string"}
FILE_B = "examples/cohort_grch37_chr.vcf"  #@param {type:"string"}

import multimerge_genomics as mmg

for label, path in (("A", FILE_A), ("B", FILE_B)):
    ref = mmg.file_reference(path)
    print(f"{label}: {path}\n   recognized assembly: {ref['assembly'] or 'not identified'}; {len(ref['sequences'])} sequences declared")
c = mmg.compare_references(FILE_A, FILE_B)
print()
print(c["verdict"])

## 6. Sort

Sorts VCF, BED, GFF/GTF or SAM by position. The sort is **stable** (records with the same position stay in the order they were in) and works for files of any size: what does not fit in memory is sorted in sorted runs on disk and merged at the end. It reads `.gz`; if the output name ends in `.gz`, the output is written compressed.

In [ ]:
#@title Sort one or more files { display-mode: "form" }
#@markdown One path, or several separated by commas (the files are merged into a single output).
INPUTS = "examples/with_defects.vcf"  #@param {type:"string"}
OUTPUT = "examples/sorted.vcf"  #@param {type:"string"}
#@markdown Chromosome order: `default` follows the standard tool for each format (header order for VCF/SAM, alphabetical for BED/GFF); `natural` is 1, 2, ..., 10, 11, ..., X, Y.
CHROMOSOME_ORDER = "natural"  #@param ["default", "natural", "lexical", "header"]
#@markdown Memory for the records, in MB (0 = automatic: half of the free memory).
MEMORY_MB = 0  #@param {type:"integer"}

import time

import multimerge_genomics as mmg

start = time.perf_counter()
records, runs, skipped = mmg.sort_genomic(
    [p.strip() for p in INPUTS.split(",") if p.strip()],
    OUTPUT,
    chrom_order=None if CHROMOSOME_ORDER == "default" else CHROMOSOME_ORDER,
    memory_mb=MEMORY_MB or None,
)
print(f"{records} records sorted in {time.perf_counter() - start:.2f} s. Written to {OUTPUT}")
if runs > 1:
    print(f"   (did not fit in the requested memory: {runs} sorted runs on disk, merged at the end)")
if skipped:
    print(f"   {skipped} lines skipped (blank or missing the minimum fields) -- run the validator to see which ones.")

## 7. BED intervals: merge and intersect

- **Merge** joins overlapping or touching intervals into one (like `bedtools merge`).
- **Intersect** answers "which parts of A fall inside B?" (like `bedtools intersect`).

The inputs do not need to be sorted or to fit in memory.

In [ ]:
#@title Merge intervals { display-mode: "form" }
INPUT = "examples/genes.bed"  #@param {type:"string"}
OUTPUT = "examples/genes_merged.bed"  #@param {type:"string"}
#@markdown Also joins intervals separated by up to this distance, in bases.
DISTANCE = 0  #@param {type:"integer"}

import multimerge_genomics as mmg

n_read, n_written = mmg.bed_merge([INPUT], OUTPUT, distance=DISTANCE)
print(f"{n_read} intervals read, {n_written} after merging. Written to {OUTPUT}")

In [ ]:
#@title Intersect two interval files { display-mode: "form" }
A = "examples/genes.bed"  #@param {type:"string"}
B = "examples/targets.bed"  #@param {type:"string"}
OUTPUT = "examples/genes_in_targets.bed"  #@param {type:"string"}
#@markdown `regions`: the parts of A that fall inside B. `unique`: the lines of A that touch B. `absent`: the lines of A that do **not** touch B.
MODE = "regions"  #@param ["regions", "unique", "absent"]

import multimerge_genomics as mmg

try:
    n = mmg.bed_intersect([A], [B], OUTPUT, mode=MODE)
    print(f"{n} lines written to {OUTPUT}")
except RuntimeError as error:
    # The classic case: one file with "chr1" and the other with "1".
    print("Could not intersect:", error)

## 8. Download a result

The files you write stay on the Colab machine and **are gone when the session ends**. Download whatever you want to keep (or write straight to Google Drive, using a path inside `/content/drive/MyDrive/`).

In [ ]:
#@title Download a file to your computer { display-mode: "form" }
FILE = "examples/sorted.vcf"  #@param {type:"string"}

try:
    from google.colab import files
    files.download(FILE)
except ImportError:
    print("Outside Colab: the file is already on your disk, at", FILE)